# Load & Extract Heart-Rate Features from Apple Health Export

This notebook reads the Apple Health `export.xml` and produces fixed-window
heart-rate features for the Health-Trajectory pipeline.

1. Clean the export (remove DOCTYPE, drop duplicate attributes)
2. Extract Heart Rate, HRV (SDNN), Resting HR and Workouts in a single pass
3. Build 15-minute HR feature windows and run a data-coverage diagnostic

> **Note on scope:** this file handles *heart-rate time series*, not ECG
> voltage waveforms. Apple exports ECG as separate CSV files in the
> `electrocardiograms/` folder, handled in a dedicated notebook.

## 1. Import necessary libraries

In [1]:
# --- Core libraries ---
import xml.etree.ElementTree as ET   # standard-library streaming XML parser
import pandas as pd
import numpy as np
import os, re, glob

print("Imports OK")

Imports OK


## 2. File path

In [2]:
# Path to the Apple Health export, and the cleaned copy we generate below.
xml_file_path = r"C:\Project\Apple Health Data\data\apple_health_export\export.xml"
temp_file     = r"C:\Project\Apple Health Data\data\apple_health_export\export_cleaned.xml"

if os.path.exists(xml_file_path):
    size_mb = os.path.getsize(xml_file_path) / (1024 * 1024)
    print(f"File found - {size_mb:.1f} MB")
else:
    raise FileNotFoundError(f"export.xml not found at: {xml_file_path}")

File found - 1056.5 MB


## 3. Load XML file and parse data

In [3]:
# ------------------------------------------------------------
# Step 1 - Clean the export.
# Apple's export.xml cannot be parsed by ElementTree as-is: it carries a
# HealthKit DOCTYPE (undefined entities) and some tags repeat an attribute,
# which ElementTree rejects. We stream the file once and (a) drop the DOCTYPE
# block, (b) keep only the FIRST occurrence of any duplicated attribute.
# ------------------------------------------------------------
print("Step 1: removing DOCTYPE and de-duplicating attributes...")

def remove_duplicate_attrs(line):
    """Keep the first value of any attribute repeated within a tag."""
    def fix_tag(match):
        tag_content = match.group(0)
        attrs = re.findall(r'(\w+)="([^"]*)"', tag_content)
        seen = {}
        for attr, value in attrs:
            if attr not in seen:
                seen[attr] = value
        tag_name = re.match(r'<(\w+)', tag_content).group(1)
        is_self_closing = tag_content.rstrip().endswith('/>')
        new_attrs = ' '.join(f'{k}="{v}"' for k, v in seen.items())
        return f'<{tag_name} {new_attrs}/>' if is_self_closing else f'<{tag_name} {new_attrs}>'
    return re.sub(r'<\w+\s+[^>]+/?>', fix_tag, line)

with open(xml_file_path, 'r', encoding='utf-8') as infile, \
     open(temp_file, 'w', encoding='utf-8') as outfile:
    in_doctype = False
    bracket_count = 0
    for line_num, line in enumerate(infile, 1):
        if '<!DOCTYPE' in line:
            in_doctype = True
        if in_doctype:
            bracket_count += line.count('[') - line.count(']')
            if ']>' in line or (bracket_count <= 0 and '>' in line):
                in_doctype = False
            continue
        if '<' in line and '="' in line:
            line = remove_duplicate_attrs(line)
        outfile.write(line)
        if line_num % 1_000_000 == 0:
            print(f"  cleaned {line_num:,} lines...")
print("Cleaning done.\n")

# ------------------------------------------------------------
# Step 2 - Single streaming pass over the cleaned file.
# We extract Heart Rate, HRV (SDNN), Resting HR and Workouts together in one
# scan instead of re-reading the multi-GB file once per data type.
#
# localname(): match tags regardless of any XML namespace, so a namespaced
#   <Record> would not silently yield zero matches.
# Memory: ElementTree keeps every finished element attached to the root, so
#   root.clear() after each top-level element is what actually keeps RAM flat
#   (elem.clear() alone does not). We clear only after a Record/Workout ends,
#   so elements with child nodes are still fully read first.
# ------------------------------------------------------------
print("Step 2: parsing cleaned file...")

def localname(tag):
    return tag.rsplit('}', 1)[-1] if '}' in tag else tag

HR_TYPE   = "HKQuantityTypeIdentifierHeartRate"
HRV_TYPE  = "HKQuantityTypeIdentifierHeartRateVariabilitySDNN"
REST_TYPE = "HKQuantityTypeIdentifierRestingHeartRate"

hr_rows, hrv_rows, rest_rows, workout_rows = [], [], [], []
total_records = 0

context = ET.iterparse(temp_file, events=("start", "end"))
_, root = next(context)                      # grab the root element for the leak fix
for event, elem in context:
    if event != "end":
        continue
    tag = localname(elem.tag)

    if tag == "Record":
        total_records += 1
        rtype = elem.get("type", "")
        if rtype == HR_TYPE:
            hr_rows.append((elem.get("startDate"), elem.get("value")))
        elif rtype == HRV_TYPE:
            hrv_rows.append((elem.get("startDate"), elem.get("value")))
        elif rtype == REST_TYPE:
            rest_rows.append((elem.get("startDate"), elem.get("value")))
        if total_records % 500_000 == 0:
            print(f"  scanned {total_records:,} records...")
        root.clear()                         # safe: Record fully parsed

    elif tag == "Workout":
        workout_rows.append((
            elem.get("workoutActivityType"),
            elem.get("startDate"),
            elem.get("endDate"),
            elem.get("duration"),            # unit is in durationUnit, usually 'min'
            elem.get("totalDistance"),       # may be empty in newer exports (moved to WorkoutStatistics)
        ))
        root.clear()                         # safe: Workout + children fully parsed

print("=" * 60)
print(f"Total records scanned : {total_records:,}")
print(f"  Heart Rate          : {len(hr_rows):,}")
print(f"  HRV (SDNN)          : {len(hrv_rows):,}")
print(f"  Resting HR          : {len(rest_rows):,}")
print(f"  Workouts            : {len(workout_rows):,}")

# Sanity check against the count from the earlier working run.
EXPECTED_TOTAL = 2_713_541
status = "MATCH" if total_records == EXPECTED_TOTAL else "DIFF - please review"
print(f"\nExpected ~{EXPECTED_TOTAL:,} records  ->  {status}")

Step 1: removing DOCTYPE and de-duplicating attributes...
  cleaned 1,000,000 lines...
  cleaned 2,000,000 lines...
  cleaned 3,000,000 lines...
Cleaning done.

Step 2: parsing cleaned file...
  scanned 500,000 records...
  scanned 1,000,000 records...
  scanned 1,500,000 records...
  scanned 2,000,000 records...
  scanned 2,500,000 records...
Total records scanned : 2,713,541
  Heart Rate          : 356,862
  HRV (SDNN)          : 4,885
  Resting HR          : 1,759
  Workouts            : 289

Expected ~2,713,541 records  ->  MATCH


## 4. Extract raw heart rate data

In [4]:
# ------------------------------------------------------------
# Turn raw Heart-Rate records into fixed 15-minute feature windows.
#
# Timezone: Apple stores local wall-clock time with an offset that switches
# between +09:30 (ACST) and +10:30 (ACDT). Parsing as UTC first and then
# converting to Australia/Adelaide yields correct, DST-consistent local
# timestamps - important for the circadian analysis later.
#
# n_samples is kept on purpose: an average from 2 readings is far less
# reliable than one from 200, and later stages use it as a quality weight.
# ------------------------------------------------------------
APPLE_FMT = "%Y-%m-%d %H:%M:%S %z"
TZ = "Australia/Adelaide"

# --- Heart Rate: raw series + 15-minute features ---
df_hr_raw = pd.DataFrame(hr_rows, columns=["datetime", "value"])
df_hr_raw["datetime"] = (pd.to_datetime(df_hr_raw["datetime"], format=APPLE_FMT, utc=True)
                           .dt.tz_convert(TZ))
df_hr_raw["value"] = df_hr_raw["value"].astype(float)
df_hr_raw = df_hr_raw.sort_values("datetime").reset_index(drop=True)

df_hr_features = (df_hr_raw.resample("15min", on="datetime")
                          .agg(avg_hr=("value", "mean"),
                               max_hr=("value", "max"),
                               min_hr=("value", "min"),
                               n_samples=("value", "count")))
df_hr_features = df_hr_features[df_hr_features["n_samples"] > 0].reset_index()

# --- HRV (SDNN): sparse & irregular, so keep it raw and left-merge the
#     per-window mean. NOTE: averaging SDNN values is a convenience summary,
#     NOT the SDNN recomputed over the whole window - state this in the paper.
if hrv_rows:
    df_hrv = pd.DataFrame(hrv_rows, columns=["datetime", "hrv_sdnn"])
    df_hrv["datetime"] = (pd.to_datetime(df_hrv["datetime"], format=APPLE_FMT, utc=True)
                            .dt.tz_convert(TZ))
    df_hrv["hrv_sdnn"] = df_hrv["hrv_sdnn"].astype(float)
    hrv_15 = (df_hrv.resample("15min", on="datetime")
                    .agg(hrv_sdnn=("hrv_sdnn", "mean")).reset_index())
    df_hr_features = df_hr_features.merge(hrv_15, on="datetime", how="left")
else:
    df_hr_features["hrv_sdnn"] = np.nan

# --- Workouts: typed timestamps; duration already in minutes ---
# NOTE: the old `df_workouts_with_hr` filter only kept workouts inside the
# overall HR time span, which is NOT the same as "workouts that have HR".
# Attaching real HR to each workout is an interval join done in the
# Context-Enrichment notebook, so here we only prepare the clean table.
df_workouts = pd.DataFrame(workout_rows,
                           columns=["type", "start", "end", "duration", "distance"])
if len(df_workouts):
    df_workouts["start_time"] = pd.to_datetime(df_workouts["start"], format=APPLE_FMT, utc=True).dt.tz_convert(TZ)
    df_workouts["end_time"]   = pd.to_datetime(df_workouts["end"],   format=APPLE_FMT, utc=True).dt.tz_convert(TZ)
    df_workouts["duration"]   = pd.to_numeric(df_workouts["duration"], errors="coerce")
    df_workouts["distance"]   = pd.to_numeric(df_workouts["distance"], errors="coerce")
    df_workouts = df_workouts.sort_values("start_time").reset_index(drop=True)

# --- Summary ---
print(f"HR feature windows : {len(df_hr_features):,}  (cols: {list(df_hr_features.columns)})")
if len(df_hr_raw):
    print(f"HR time range      : {df_hr_raw['datetime'].min()}  ->  {df_hr_raw['datetime'].max()}")
    print(f"HR value range     : {df_hr_raw['value'].min():.0f} - {df_hr_raw['value'].max():.0f} bpm")
print(f"Workouts           : {len(df_workouts):,}")
df_hr_features.head()

HR feature windows : 104,565  (cols: ['datetime', 'avg_hr', 'max_hr', 'min_hr', 'n_samples', 'hrv_sdnn'])
HR time range      : 2017-11-27 10:47:34+10:30  ->  2022-09-23 10:59:44+09:30
HR value range     : 41 - 199 bpm
Workouts           : 289


,datetime,avg_hr,max_hr,min_hr,n_samples,hrv_sdnn
0,2017-11-27 10:45:00+10:30,82.000000,82.0,82.0,1,NaN
1,2017-11-27 11:00:00+10:30,73.500000,74.0,72.0,4,NaN
2,2017-11-27 11:15:00+10:30,76.833333,80.0,74.0,12,NaN
3,2017-11-27 11:30:00+10:30,72.222222,76.0,68.0,9,NaN
4,2017-11-27 11:45:00+10:30,73.000000,74.0,72.0,3,NaN


## 5. Inspect workout GPS routes (optional)

In [5]:
# Optional: peek at one workout-route GPX file to see its structure
# (track points = lat / lon / elevation / time). GPS is only recorded during
# workouts, so most non-workout windows will have no location - that is
# expected and handled in the Context-Enrichment notebook.
gpx_folder = r"C:\Project\Apple Health Data\data\apple_health_export\workout-routes"
gpx_files = glob.glob(os.path.join(gpx_folder, "*.gpx"))
print(f"Found {len(gpx_files)} GPX route files")

if gpx_files:
    sample = gpx_files[0]
    root = ET.parse(sample).getroot()
    trkpts = [e for e in root.iter() if e.tag.endswith("trkpt")]   # namespace-agnostic
    print(f"Sample file: {os.path.basename(sample)} - {len(trkpts)} track points")
    if trkpts:
        p = trkpts[0]
        print("First point attributes:", dict(p.attrib))
        for child in p:
            print(f"   {child.tag.split('}')[-1]}: {child.text}")

Found 177 GPX route files
Sample file: route_2018-01-07_4.49pm.gpx - 8916 track points
First point attributes: {'lon': '138.597218', 'lat': '-35.004800'}
   ele: 138.675110
   time: 2018-01-07T03:39:50Z
   extensions: None


## 6. Data-coverage diagnostic (cosinor feasibility)

In [6]:
# ------------------------------------------------------------
# Is the record dense enough - and does it cover the overnight hours - for a
# circadian (cosinor) baseline? For cosinor the risk is NOT total volume but
# (a) uneven density over the years and (b) missing 00:00-05:00 (watch charged
# overnight). The deciding numbers are the night-coverage ratio and the
# longest run of well-covered days.
# ------------------------------------------------------------
df = df_hr_raw.copy()
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour

# (1) Monthly record density - spot gaps / bursts across the years
print("Records per month (describe):")
print(df.set_index("datetime").resample("ME").size().describe())

# (2) Coverage by hour-of-day - pay attention to hours 0..5
print("\nRecords by hour-of-day:")
print(df.groupby("hour").size())

# (3) Per-day quality: distinct hours covered + whether night is present
per_day = df.groupby("date").agg(
    n=("value", "size"),
    hours=("hour", "nunique"),
    has_night=("hour", lambda h: h.between(0, 5).any()))
print(f"\nDays with >=18 hours covered : {(per_day['hours'] >= 18).mean():.1%}")
print(f"Days with night data (0-5h)  : {per_day['has_night'].mean():.1%}")

# (4) Longest run of consecutive 'good' days (>=18h AND night present)
good = pd.to_datetime(pd.Series(sorted(
    per_day[(per_day["hours"] >= 18) & per_day["has_night"]].index)))
if len(good):
    run = (good.diff().dt.days != 1).cumsum()
    print(f"Longest run of good days     : {int(good.groupby(run).size().max())}")
else:
    print("Longest run of good days     : 0 (no fully-covered days)")

Records per month (describe):
count       59.000000
mean      6048.508475
std       1198.283620
min        837.000000
25%       5477.000000
50%       5930.000000
75%       6570.500000
max      10985.000000
dtype: float64

Records by hour-of-day:
hour
0      4706
1      3504
2      3628
3      3247
4      3268
5      3214
6      2534
7      4427
8     11141
9     16824
10    22035
11    22744
12    22279
13    23040
14    22002
15    22379
16    23022
17    22768
18    22540
19    22379
20    22937
21    22235
22    18777
23    11232
dtype: int64

Days with >=18 hours covered : 11.4%
Days with night data (0-5h)  : 27.1%
Longest run of good days     : 9


## 7. Persist the output contract

File 1 hands three tables to the rest of the pipeline. We save them to `data/processed/` as parquet (tz-aware datetimes survive parquet) so File 2-4 load them instantly instead of re-parsing the 1 GB `export.xml` every run.

**Output contract** (do not rename these columns downstream):
- `df_hr_raw`     -> `hr_raw.parquet`      `[datetime(tz), value]`
- `df_hr_features`-> `hr_features.parquet` `[datetime(tz), avg_hr, max_hr, min_hr, n_samples, hrv_sdnn]`
- `df_workouts`   -> `workouts.parquet`    `[type, start_time(tz), end_time(tz), duration, distance]`

In [7]:
# --- Persist output contract to data/processed (parquet) ---
# tz-aware timestamps are preserved by parquet (pyarrow). n_samples travels with
# the features because File 3 needs it as the WLS baseline weight.
import os

PROC_DIR = r"C:\Project\Apple Health Data\data\processed"
os.makedirs(PROC_DIR, exist_ok=True)

df_hr_raw.to_parquet(os.path.join(PROC_DIR, "hr_raw.parquet"), index=False)
df_hr_features.to_parquet(os.path.join(PROC_DIR, "hr_features.parquet"), index=False)
# keep only the typed workout columns (drop the raw string helpers)
_wcols = ["type", "start_time", "end_time", "duration", "distance"]
df_workouts[_wcols].to_parquet(os.path.join(PROC_DIR, "workouts.parquet"), index=False)

print("Saved to", PROC_DIR)
for f in ("hr_raw.parquet", "hr_features.parquet", "workouts.parquet"):
    p = os.path.join(PROC_DIR, f)
    print(f"  {f:22s} {os.path.getsize(p)/1e6:6.1f} MB")

Saved to C:\Project\Apple Health Data\data\processed
  hr_raw.parquet            3.3 MB
  hr_features.parquet       1.6 MB
  workouts.parquet          0.0 MB
